In [1]:
%run -i ../python_scripts/nb_setup.py
from joblib import cpu_count

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060


In [2]:
n_cpus = cpu_count()

#### Choose the dataset you want to use :
* must be data the model did not see at training
* first column : __y_true__ $\in \{0,1\}$ are the samples labels  
* second column : __y_pred__ are the predicted class of samples ($\in \{0,1\}$ too)
* third column : __kappa_f__ is confidence function for each sample, $\in \mathbb{R}$

In [3]:
sgp_df = pickle.load(open("../experiments/CIFAR/sgp_set_cnn", "rb"))
print(
    "N =",
    sgp_df.shape[0],
    "Proportion of 1s: ",
    np.round(sgp_df.y_true.sum() / sgp_df.shape[0], 2),
)
sgp_df.head(3)

N = 40000 Proportion of 1s:  0.1


,y_true,y_pred,kappa
0,0.0,0.0,0.968736
1,0.0,1.0,0.870544
2,0.0,0.0,0.882295


In [4]:
deltas = [1e-4, 1e-3, 1e-2, 1e-1]
num_seed = 500
theta_min, theta_max = 0.5, 1  # Sn-independent grid
num_targets = 20  # number of targets r* to consider when drawing metric/coverage of metric/theta curves
metric_targets = np.linspace(0, 1, num=num_targets)

In [5]:
results = []

### __0/1 risk__

In [6]:
for delta in deltas:
    failures_01 = list(
        Parallel(n_jobs=n_cpus - 2)(
            delayed(
                lambda seed: run_one_seed(
                    s=seed,
                    delta=delta,
                    sgp_df=sgp_df,
                    metric="standard",
                    metric_targets=metric_targets,
                    mode="dicho",
                    theta_min=theta_min,
                    theta_max=theta_max,
                )
            )(s)
            for s in range(num_seed)
        )
    )
    failure_rate = sum(failures_01) / (num_seed * num_targets)
    results.append({"metric": "01", "delta": delta, "failure rate": failure_rate})
    print("failure rate:", failure_rate)

failure rate: 0.0
failure rate: 0.0
failure rate: 0.0
failure rate: 0.0


### __FP risk__

In [7]:
for delta in deltas:
    failures_fp = list(
        Parallel(n_jobs=n_cpus - 2)(
            delayed(
                lambda seed: run_one_seed(
                    s=seed,
                    delta=delta,
                    sgp_df=sgp_df,
                    metric="FP",
                    metric_targets=metric_targets,
                    mode="dicho",
                    theta_min=theta_min,
                    theta_max=theta_max,
                )
            )(s)
            for s in range(num_seed)
        )
    )
    failure_rate = sum(failures_fp) / (num_seed * num_targets)
    results.append({"metric": "FP risk", "delta": delta, "failure rate": failure_rate})
    print("failure rate:", failure_rate)

failure rate: 0.0
failure rate: 0.0
failure rate: 0.0
failure rate: 0.0


### __FN risk__

In [8]:
for delta in deltas:
    failures_fn = list(
        Parallel(n_jobs=n_cpus - 2)(
            delayed(
                lambda seed: run_one_seed(
                    s=seed,
                    delta=delta,
                    sgp_df=sgp_df,
                    metric="FN",
                    metric_targets=metric_targets,
                    mode="dicho",
                    theta_min=theta_min,
                    theta_max=theta_max,
                )
            )(s)
            for s in range(num_seed)
        )
    )
    failure_rate = sum(failures_fn) / (num_seed * num_targets)
    results.append({"metric": "FN risk", "delta": delta, "failure rate": failure_rate})
    print("failure rate:", failure_rate)

failure rate: 0.0
failure rate: 0.0
failure rate: 0.0
failure rate: 0.0


### __FPR__

In [9]:
for delta in deltas:
    failures_fpr = list(
        Parallel(n_jobs=n_cpus - 2)(
            delayed(
                lambda seed: run_one_seed(
                    s=seed,
                    delta=delta,
                    sgp_df=sgp_df,
                    metric="FPR",
                    metric_targets=metric_targets,
                    mode="greedy",
                    theta_min=theta_min,
                    theta_max=theta_max,
                )
            )(s)
            for s in range(num_seed)
        )
    )
    failure_rate = sum(failures_fpr) / (num_seed * num_targets)
    results.append({"metric": "FPR", "delta": delta, "failure rate": failure_rate})
    print("failure rate:", failure_rate)

failure rate: 0.0
failure rate: 0.0
failure rate: 0.0
failure rate: 0.0


### __FNR__

In [10]:
for delta in deltas:
    failures_fnr = list(
        Parallel(n_jobs=n_cpus - 2)(
            delayed(
                lambda seed: run_one_seed(
                    s=seed,
                    delta=delta,
                    sgp_df=sgp_df,
                    metric="FNR",
                    metric_targets=metric_targets,
                    mode="greedy",
                    theta_min=theta_min,
                    theta_max=theta_max,
                )
            )(s)
            for s in range(num_seed)
        )
    )
    failure_rate = sum(failures_fnr) / (num_seed * num_targets)
    results.append({"metric": "FNR", "delta": delta, "failure rate": failure_rate})
    print("failure rate:", failure_rate)

failure rate: 0.0
failure rate: 0.0
failure rate: 0.0
failure rate: 0.0


In [11]:
pickle.dump(results, open("failure_rates", "wb"))